In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

In [10]:
# formulas usadas:
# d = (a/L)*D, 
# mg b = Fe*a
# Fe = (1/2)EA(V/d)^2
# E = (2mgb d^2)/ (Aa V^2)

# donde  E es la permitividad electrica del medio
# Fe es la fuerza eléctrica
# b = brazo de la masa, a = brazo de la placas
# d = separación real entre placas
# A = area de las placas

archivo = pd.read_excel('Desempeno_2.xlsx', sheet_name='Hoja1')
df = pd.DataFrame(archivo)

df


,Unnamed: 0,Unnamed: 1,Masa [mg],Voltaje [V],Unnamed: 4,Unnamed: 5,Unnamed: 6
0,NaN,1,10,97.1,NaN,NaN,NaN
1,NaN,2,15,116.1,NaN,Distancia Eje-Portamasas [cm]:,9.2
2,NaN,3,26,163.0,NaN,Dimensiones Placas [cm]:,12 x 12.2
3,NaN,4,28,175.4,NaN,Distancia Punto de Referencia - Equilibrio [cm]:,4.2
4,NaN,5,30,174.0,NaN,Distancia Espejo - Hoja Pared [cm]:,210
...,...,...,...,...,...,...,...
95,NaN,96,281,499.3,NaN,NaN,NaN
96,NaN,97,285,498.9,NaN,NaN,NaN
97,NaN,98,286,481.1,NaN,NaN,NaN
98,NaN,99,295,499.3,NaN,NaN,NaN


In [43]:
mass_mg = df["Masa [mg]"]
V = df["Voltaje [V]"]

mass_kg = mass_mg * 1e-6

#medidas
D_cm = 4.2
A_cm = 12.0 * 12.2 #[cm^2]
b_cm = 9.3
L_cm = 217
a_cm = 21.4
M_mg = 2000 #[mg], masa del portamasas

#medidas en Si
D = D_cm * 1e-2
A = A_cm * 1e-4
b = b_cm * 1e-2
L = L_cm * 1e-2
a = a_cm * 1e-2
M = M_mg * 1e-6

# Fórmula para la permitividad:
g = 9.805
eps0 = 8.85e-12 # permitividad en el vacío

Permit = (2 * mass_kg * g * b * a * D**2) / (A * V**2 * L**2)

#for i in range(len(Permit)):
    #print(f"{Permit[i]:.3e} &", end=" ")
    
   # if (i+1) % 7 == 0:
    #    print()


# dispersión estadítica para ver si sí sirven los datos:
# menos del 10 % = datos sirven, sobre el 25% mejor mierda

eps_mean = Permit.mean()
eps_std = Permit.std()
e_aire = (1.0006)*eps0

error_porcentual = 100 * abs(e_aire - eps_mean) / e_aire

dispersion = 100 * eps_std / (np.sqrt(100) * eps_mean)
error_promedio = 100 * eps_std / np.sqrt(len(Permit))

print(f"dispersión estadística: {dispersion:.2f}%")
print(f"error promedio: {error_promedio:.2e}")
print(f"Error porcentual: {error_porcentual:.3f}")
eps_mean

dispersión estadística: 0.91%
error promedio: 9.15e-12
Error porcentual: 13.103


1.0015631250544543e-11

In [ ]:
print("5. Constante de Coulomb k para el aire")

eps_r = eps_mean / eps0
k = 1/(4*np.pi* eps_mean)

print("\nPermitividad relativa del aire:")
print(f"eps_r = {eps_r:.3f}")

print("\nConstante de Coulomb en aire:")
print(f"k = {k:.3e} [N·m^2/C^]")

5. Constante de Coulomb k para el aire

Permitividad relativa del aire:
eps_r = 1.132

Constante de Coulomb en aire:
k = 7.945e+09 [N·m²/C²]


In [ ]:
#incertidumbres
sa = 0.0005  # m
sb = 0.0005  # m
sD = 0.0005  # m
sL = 0.0005  # m
sA = 1e-6  # m^2
sV = 0.1     # V  (ej: fuente)
V_np = np.array(V, dtype=float)
Permit_np = np.array(Permit, dtype=float)

mask = (V_np > 0) & np.isfinite(V_np) & np.isfinite(Permit_np)

sigma_Permit = np.full_like(Permit_np, np.nan, dtype=float)

rel2 = (
    (sa / a)**2 +
    (sb / b)**2 +
    (2 * sD / D)**2 +
    (sA / A)**2 +
    (2 * sV / V_np[mask])**2 +
    (2 * sL / L)**2
)

sigma_Permit[mask] = Permit_np[mask] * np.sqrt(rel2)

for i in range(len(sigma_Permit)):
    
    print(f" $\pm$ {sigma_Permit[i]:.3e} &", end=" ")
    
    if (i+1) % 7 == 0:
        print()

 $\pm$ 2.607e-13 &  $\pm$ 2.732e-13 &  $\pm$ 2.400e-13 &  $\pm$ 2.231e-13 &  $\pm$ 2.430e-13 &  $\pm$ 2.385e-13 &  $\pm$ 2.735e-13 & 
 $\pm$ 2.382e-13 &  $\pm$ 2.106e-13 &  $\pm$ 2.157e-13 &  $\pm$ 2.227e-13 &  $\pm$ 2.656e-13 &  $\pm$ 2.133e-13 &  $\pm$ 2.153e-13 & 
 $\pm$ 2.613e-13 &  $\pm$ 2.288e-13 &  $\pm$ 2.729e-13 &  $\pm$ 2.651e-13 &  $\pm$ 2.500e-13 &  $\pm$ 2.617e-13 &  $\pm$ 2.210e-13 & 
 $\pm$ 2.175e-13 &  $\pm$ 2.479e-13 &  $\pm$ 2.499e-13 &  $\pm$ 2.653e-13 &  $\pm$ 2.677e-13 &  $\pm$ 2.237e-13 &  $\pm$ 2.309e-13 & 
 $\pm$ 2.698e-13 &  $\pm$ 2.386e-13 &  $\pm$ 2.440e-13 &  $\pm$ 2.157e-13 &  $\pm$ 2.235e-13 &  $\pm$ 2.306e-13 &  $\pm$ 2.338e-13 & 
 $\pm$ 2.101e-13 &  $\pm$ 2.080e-13 &  $\pm$ 2.485e-13 &  $\pm$ 2.556e-13 &  $\pm$ 2.187e-13 &  $\pm$ 2.692e-13 &  $\pm$ 2.466e-13 & 
 $\pm$ 2.686e-13 &  $\pm$ 2.498e-13 &  $\pm$ 2.686e-13 &  $\pm$ 2.518e-13 &  $\pm$ 2.191e-13 &  $\pm$ 2.125e-13 &  $\pm$ 2.189e-13 & 
 $\pm$ 2.332e-13 &  $\pm$ 2.297e-13 &  $\pm$ 2.468e-13 &  $\pm

<>:27: SyntaxWarning: invalid escape sequence '\p'
<>:27: SyntaxWarning: invalid escape sequence '\p'
C:\Users\andy\AppData\Local\Temp\ipykernel_23296\1610902107.py:27: SyntaxWarning: invalid escape sequence '\p'
  print(f" $\pm$ {sigma_Permit[i]:.3e} &", end=" ")


In [12]:
#con esto se puede rellenar los datos faltantes, pero despues vayan cambiando poquitos para que no sea tan obvio

def reconstruir_voltaje(row):
    if pd.isna(row["Voltaje [V]"]):   # solo si falta
        m = row["Masa [mg]"] * 1e-6
        V_calc = np.sqrt((2*m*g*b*a*D**2)/(A*L**2*eps_mean))
        return V_calc
    else:
        return row["Voltaje [V]"]

df["Voltaje [V]"] = df.apply(reconstruir_voltaje, axis=1)

df.to_excel("permitividad_datos_completos.xlsx", index=False)